# ZARX-1B Training Notebook (Kaggle)
1B parameter code + ARC reasoning model trained from scratch.

**GPU:** T4/P100 (16GB VRAM) | **Session:** 9 hours max

## Setup
1. Create a new notebook on Kaggle
2. Settings → Accelerator → GPU T4 x2 or P100
3. Internet: ON (to download packages and sync checkpoints)
4. Run cells in order

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch>=2.1.0 bitsandbytes wandb huggingface_hub
!pip install -q datasets tokenizers accelerate datasketch
!pip install -q flash-attn --no-build-isolation
print('Dependencies installed!')

In [ ]:
#@title 2. Login to HuggingFace & W&B
import os

HF_TOKEN = '' #@param {type:"string"}
if HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}

WANDB_KEY = '' #@param {type:"string"}
if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)

print('Logins complete!')

In [ ]:
#@title 3. Clone ZARX-1B Repository
%cd /kaggle/working
!rm -rf ZARX-1B
!git clone https://github.com/YOURUSERNAME/ZARX-1B.git
%cd ZARX-1B
print('Repository cloned!')

In [ ]:
#@title 4. Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
#@title 5. Start Training (auto-resumes from HuggingFace checkpoints)
HF_REPO = '' #@param {type:"string"}
HF_TOKEN = '' #@param {type:"string"}

# On Kaggle, data should be pre-uploaded as a Kaggle dataset
# or downloaded via the download script

!python src/train.py \
    --model_config configs/model_config.json \
    --tokenizer_path configs/tokenizer.json \
    --data_path /kaggle/working/ZARX-1B/data/processed \
    --micro_batch_size 1 \
    --gradient_accumulation_steps 32 \
    --learning_rate 3e-4 \
    --warmup_steps 2000 \
    --total_tokens 10000000000 \
    --use_8bit_adam \
    --checkpoint_dir /kaggle/working/checkpoints \
    --hf_repo_id {HF_REPO} \
    --hf_token {HF_TOKEN} \
    --save_every_local 100 \
    --save_every_hf 1000 \
    --log_every_steps 10 \
    --wandb_project zarx-1b \
    --output_dir /kaggle/working/zarx-1b-final

In [ ]:
#@title 6. Save Final Checkpoint to Kaggle Output
import shutil
shutil.copy('/kaggle/working/checkpoints/checkpoint-latest.pt', '/kaggle/working/zarx-final-checkpoint.pt')
print('Final checkpoint saved to Kaggle output!')